# Day 5 — Classical Clickbait Baseline

**Lesson focus:** Review TF-IDF and Logistic Regression.

**Required evidence:** Train a clickbait baseline and save its metrics.

Labels in this practice task: `0 = factual`, `1 = clickbait`.

## 1. Imports

In [1]:
import json
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

## 2. Practice clickbait dataset

The dataset is intentionally small and balanced so we can concentrate on the classical baseline workflow. Later projects will replace this with a real dataset.

In [2]:
factual_texts = [
    'Government publishes annual budget report',
    'Election commission announces voting schedule',
    'University releases new admission guidelines',
    'Football team wins league match two one',
    'Central bank keeps interest rate unchanged',
    'City council approves road repair project',
    'Hospital opens new emergency care unit',
    'Weather office forecasts rain this weekend',
    'Company reports quarterly revenue growth',
    'School board changes examination schedule',
    'Researchers publish results of health study',
    'Court announces decision in public case',
    'National team names squad for tournament',
    'Airport adds two international flight routes',
    'Ministry releases updated education policy',
    'Local market opens at eight tomorrow',
    'Police issue traffic advisory for festival',
    'Railway announces revised ticket prices',
    'Museum opens exhibition on regional history',
    'Scientists record seasonal river level changes',
]

clickbait_texts = [
    'You will not believe what happened next',
    'This shocking secret has everyone talking',
    'What this celebrity did next stunned fans',
    'The truth they never wanted you to know',
    'Unbelievable moment caught everyone by surprise',
    'This one trick will change your life',
    'Everyone is talking about this shocking reveal',
    'You need to see what happened next',
    'The hidden reason behind this unbelievable story',
    'Fans cannot believe this surprise announcement',
    'What happened next left viewers speechless',
    'This secret detail changes everything',
    'Nobody expected this shocking turn of events',
    'Wait until you see the final result',
    'The real story will completely surprise you',
    'One surprising fact everyone needs to know',
    'This incredible discovery has people amazed',
    'Can you guess what happened after this',
    'The answer will leave you completely shocked',
    'What they found next was truly unbelievable',
]

df = pd.DataFrame({
    'text': factual_texts + clickbait_texts,
    'label': [0] * len(factual_texts) + [1] * len(clickbait_texts),
})

print('Dataset shape:', df.shape)
print(df['label'].value_counts().sort_index())

Dataset shape: (40, 2)
label
0    20
1    20
Name: count, dtype: int64


## 3. Stratified train / validation / test split

We keep approximately 70% for training, 15% for validation, and 15% for final testing. `stratify` preserves the class proportions.

In [3]:
X = df['text']
y = df['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50,
    random_state=RANDOM_STATE, stratify=y_temp
)

print(f'Train: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}')
print('Train clickbait rate:', round(y_train.mean(), 3))
print('Validation clickbait rate:', round(y_val.mean(), 3))
print('Test clickbait rate:', round(y_test.mean(), 3))

Train: 28 | Validation: 6 | Test: 6
Train clickbait rate: 0.5
Validation clickbait rate: 0.5
Test clickbait rate: 0.5


## 4. TF-IDF + Logistic Regression

TF-IDF converts text into weighted numeric features. Logistic Regression then learns weights that separate factual and clickbait examples. The pipeline ensures TF-IDF is fitted only on the training split.

## 5. Compare `(1,1)`, `(1,2)`, and `(1,3)` n-grams

We compare unigram, unigram+bigram, and unigram+bigram+trigram representations using **validation Macro-F1**. If two choices tie, we select the simpler one with fewer features. This is why `(1,3)` is not automatically better even though it can capture a phrase such as `you will not believe`.

In [4]:
candidate_ranges = [(1, 1), (1, 2), (1, 3)]
candidate_models = {}
comparison_rows = []

for ngram_range in candidate_ranges:
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=ngram_range, lowercase=True)),
        ('logreg', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])
    pipeline.fit(X_train, y_train)
    val_pred = pipeline.predict(X_val)

    macro_f1 = f1_score(y_val, val_pred, average='macro')
    feature_count = len(pipeline.named_steps['tfidf'].vocabulary_)
    candidate_models[ngram_range] = pipeline
    comparison_rows.append({
        'ngram_range': str(ngram_range),
        'validation_accuracy': accuracy_score(y_val, val_pred),
        'validation_macro_f1': macro_f1,
        'feature_count': feature_count,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,ngram_range,validation_accuracy,validation_macro_f1,feature_count
0,"(1, 1)",0.833333,0.828571,136
1,"(1, 2)",0.833333,0.828571,278
2,"(1, 3)",0.833333,0.828571,395


## 6. Select the baseline from validation results

Selection rule: highest validation Macro-F1 first, then fewer TF-IDF features as the tie-breaker. The test set is **not** used for this decision.

In [5]:
best_row = comparison_df.sort_values(
    ['validation_macro_f1', 'feature_count'],
    ascending=[False, True],
).iloc[0]

best_ngram = tuple(int(v.strip()) for v in best_row['ngram_range'].strip('()').split(','))
best_model = candidate_models[best_ngram]

print('Selected n-gram range:', best_ngram)
print('Validation Macro-F1:', round(best_row['validation_macro_f1'], 3))
print('Number of TF-IDF features:', int(best_row['feature_count']))

Selected n-gram range: (1, 1)
Validation Macro-F1: 0.829
Number of TF-IDF features: 136


## 7. Final test metrics

Now—and only now—we evaluate the selected baseline on the untouched test split.

In [6]:
test_pred = best_model.predict(X_test)

test_metrics = {
    'accuracy': accuracy_score(y_test, test_pred),
    'precision': precision_score(y_test, test_pred, zero_division=0),
    'recall': recall_score(y_test, test_pred, zero_division=0),
    'f1': f1_score(y_test, test_pred, zero_division=0),
    'macro_f1': f1_score(y_test, test_pred, average='macro', zero_division=0),
}

print('Final test metrics:')
for metric, value in test_metrics.items():
    print(f'{metric:>10}: {value:.3f}')

print('\nClassification report:')
print(classification_report(y_test, test_pred, target_names=['factual', 'clickbait'], digits=3, zero_division=0))

Final test metrics:
  accuracy: 0.833
 precision: 0.750
    recall: 1.000
        f1: 0.857
  macro_f1: 0.829

Classification report:
              precision    recall  f1-score   support

     factual      1.000     0.667     0.800         3
   clickbait      0.750     1.000     0.857         3

    accuracy                          0.833         6
   macro avg      0.875     0.833     0.829         6
weighted avg      0.875     0.833     0.829         6



## 8. Required evidence — save metrics

The saved JSON records the selected TF-IDF setting, split sizes, validation score, and final test metrics. In this repository layout, running the notebook from `notebooks/` saves it to `../results/day_05_baseline_metrics.json`.

In [7]:
metrics_payload = {
    'model': 'TF-IDF + Logistic Regression',
    'selected_ngram_range': list(best_ngram),
    'selection_metric': 'validation_macro_f1',
    'validation_macro_f1': float(best_row['validation_macro_f1']),
    'split_sizes': {
        'train': len(X_train),
        'validation': len(X_val),
        'test': len(X_test),
    },
    'test_metrics': {key: float(value) for key, value in test_metrics.items()},
}

results_dir = Path('../results')
results_dir.mkdir(parents=True, exist_ok=True)
metrics_path = results_dir / 'day_05_baseline_metrics.json'

with metrics_path.open('w', encoding='utf-8') as file:
    json.dump(metrics_payload, file, indent=2)

print('Metrics saved to:', metrics_path)
metrics_payload

Metrics saved to: ../results/day_05_baseline_metrics.json


{'model': 'TF-IDF + Logistic Regression',
 'selected_ngram_range': [1, 1],
 'selection_metric': 'validation_macro_f1',
 'validation_macro_f1': 0.8285714285714285,
 'split_sizes': {'train': 28, 'validation': 6, 'test': 6},
 'test_metrics': {'accuracy': 0.8333333333333334,
  'precision': 0.75,
  'recall': 1.0,
  'f1': 0.8571428571428571,
  'macro_f1': 0.8285714285714285}}

## 9. Verification

In [8]:
assert len(X_train) + len(X_val) + len(X_test) == len(df)
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
assert metrics_path.exists()
assert 0.0 <= metrics_payload['test_metrics']['macro_f1'] <= 1.0
assert best_ngram in candidate_ranges

print('All Day 5 classical-baseline checks passed successfully!')

All Day 5 classical-baseline checks passed successfully!


## Independent practice

1. Explain why validation Macro-F1—not test performance—is used to choose the n-gram range.
2. Compare the feature counts for `(1,1)`, `(1,2)`, and `(1,3)`.
3. Explain why `(1,3)` may capture richer phrases but still fail to improve validation performance.
4. Change Logistic Regression `C` from `1.0` to `0.5` and compare validation Macro-F1.
5. Open the saved JSON metrics file and explain each field in your own words.